# Workshop 3 — ETL Streaming with Apache Kafka
## Step 3 & 4: Feature Engineering + Model Training

**Course:** ETL (G01) — Data Engineering and Artificial Intelligence  
**Input:** `data/processed/happiness_unified.csv`  
**Output:** `models/model.pkl`

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded successfully ✓')

---
## 1. Load Unified Dataset

In [ ]:
df = pd.read_csv('../data/processed/happiness_unified.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
display(df.head())

---
## 2. Feature Engineering

### 2.1 Feature Selection

**Target:** `happiness_score`

**Selected features:**

| Feature | Justification |
|---|---|
| `gdp` | High correlation with happiness (>0.8 in EDA) |
| `family` | Social support — strong predictor |
| `health` | Life expectancy — high correlation |
| `freedom` | Freedom of choice — moderate-high correlation |
| `generosity` | Included for model completeness |
| `corruption` | Institutional perception — moderate correlation |

**Discarded features:**
- `country`: high-cardinality categorical variable, not useful for simple regression
- `year`: not a causal predictor of happiness

**No target leakage:** no feature is derived directly from happiness_score.

In [ ]:
FEATURES = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
TARGET = 'happiness_score'

# Verify all features exist
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f'⚠️ Missing features: {missing}')
else:
    print('All features available ✓')

# Final dataset for ML
df_ml = df[FEATURES + [TARGET]].dropna()
print(f'Records for training: {df_ml.shape[0]}')
display(df_ml.describe())

### 2.2 Relationship Visualizations

In [ ]:
# Correlation of each feature with the target
correlations = df_ml.corr()[TARGET].drop(TARGET).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
correlations.plot(kind='bar', color='steelblue')
plt.title('Feature Correlation with Happiness Score')
plt.ylabel('Correlation')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\nCorrelations:')
print(correlations)

---
## 3. Train/Test Split (70/30)

In [ ]:
X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f'Train: {X_train.shape[0]} records ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} records ({X_test.shape[0]/len(X)*100:.1f}%)')

---
## 4. Model Training and Evaluation

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2   = r2_score(y_test, y_pred)

    print(f'\n{"="*40}')
    print(f'Model: {name}')
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R²:   {r2:.4f}')

    return {'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2, 'y_pred': y_pred}

models = [
    ('Linear Regression',  LinearRegression()),
    ('Decision Tree',      DecisionTreeRegressor(random_state=42)),
    ('Random Forest',      RandomForestRegressor(n_estimators=100, random_state=42)),
]

results = []
for name, model in models:
    res = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(res)

### 4.1 Model Comparison

In [ ]:
metrics_df = pd.DataFrame([{
    'Model': r['name'],
    'MAE': round(r['mae'], 4),
    'RMSE': round(r['rmse'], 4),
    'R²': round(r['r2'], 4)
} for r in results])

display(metrics_df)

# Comparison chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    ax.bar(metrics_df['Model'], metrics_df[metric], color=['steelblue', 'coral', 'seagreen'])
    ax.set_title(metric)
    ax.set_xticklabels(metrics_df['Model'], rotation=15, ha='right')

plt.suptitle('Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Final Model Selection and Serialization

**Selection criterion:** best R² with lowest RMSE.  
This workshop does not require optimization — the model with the best baseline metrics is selected.

In [ ]:
# Select the best model by R²
best = max(results, key=lambda x: x['r2'])
print(f'Selected model: {best["name"]}')
print(f'  R²:   {best["r2"]:.4f}')
print(f'  MAE:  {best["mae"]:.4f}')
print(f'  RMSE: {best["rmse"]:.4f}')

In [ ]:
# Save the model and the feature list (required by the consumer)
os.makedirs('../models', exist_ok=True)

model_artifact = {
    'model': best['model'],
    'features': FEATURES
}

with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print('Model saved to ../models/model.pkl ✓')
print(f'Model features: {FEATURES}')

---
## 6. Visualization: Predicted vs Actual

In [ ]:
y_pred_best = best['y_pred']

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Happiness Score')
plt.ylabel('Predicted Happiness Score')
plt.title(f'Predicted vs Actual — {best["name"]}')
plt.tight_layout()
plt.show()

In [ ]:
# Prediction error distribution
errors = y_test.values - y_pred_best

plt.figure(figsize=(8, 4))
sns.histplot(errors, bins=30, kde=True, color='coral')
plt.axvline(0, color='black', linestyle='--')
plt.title('Prediction Error Distribution')
plt.xlabel('Error (actual - predicted)')
plt.tight_layout()
plt.show()

print(f'Mean error: {errors.mean():.4f}')
print(f'Error standard deviation: {errors.std():.4f}')

---
## 7. Saved Model Verification

In [ ]:
# Load and test the saved model
with open('../models/model.pkl', 'rb') as f:
    artifact = pickle.load(f)

loaded_model    = artifact['model']
loaded_features = artifact['features']

# Test with a sample event (same format as the Kafka producer)
test_event = {
    'country': 'Colombia',
    'year': 2019,
    'gdp': 1.2,
    'family': 0.8,
    'health': 0.9,
    'freedom': 0.6,
    'generosity': 0.3,
    'corruption': 0.1,
    'actual_happiness_score': 6.2
}

input_df   = pd.DataFrame([[test_event[f] for f in loaded_features]], columns=loaded_features)
prediction = loaded_model.predict(input_df)[0]

print('Model loaded successfully ✓')
print(f'Expected features: {loaded_features}')
print(f'\nTest event: Colombia 2019')
print(f'  Actual:    {test_event["actual_happiness_score"]}')
print(f'  Predicted: {prediction:.4f}')
print(f'  Error:     {abs(test_event["actual_happiness_score"] - prediction):.4f}')